In [3]:
!pip install evidently

In [4]:
from evidently import Report

from evidently.presets import DataDriftPreset

In [7]:
# ============================================================
# DAY 12 — DATA DRIFT MONITORING (Stable Version)
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import ks_2samp

# ============================================================
# LOAD DATA
# ============================================================

sales = pd.read_csv(
    "data/processed/sales_cleaned.csv",
    parse_dates=['InvoiceDate']
)

print("Sales Shape:", sales.shape)

# ============================================================
# REFERENCE VS CURRENT WINDOWS
# ============================================================

ref_end = (
    sales['InvoiceDate'].min()
    + pd.Timedelta(days=180)
)

cur_start = (
    sales['InvoiceDate'].max()
    - pd.Timedelta(days=30)
)

reference = sales[
    sales['InvoiceDate'] <= ref_end
]

current = sales[
    sales['InvoiceDate'] >= cur_start
]

print("\nReference Rows:", len(reference))
print("Current Rows  :", len(current))

# ============================================================
# FEATURES TO MONITOR
# ============================================================

features = [
    'Quantity',
    'UnitPrice',
    'TotalPrice'
]

# ============================================================
# DRIFT DETECTION
# ============================================================

drift_results = []

for col in features:

    ref_data = reference[col].dropna()

    cur_data = current[col].dropna()

    # Kolmogorov-Smirnov test

    stat, p_value = ks_2samp(
        ref_data,
        cur_data
    )

    drift_detected = p_value < 0.05

    drift_results.append({
        "feature": col,
        "p_value": p_value,
        "drift_detected": drift_detected
    })

# ============================================================
# SUMMARY
# ============================================================

n_drifted = sum(
    r["drift_detected"]
    for r in drift_results
)

share = n_drifted / len(features)

# ============================================================
# ALERT WRAPPER
# ============================================================

threshold = 0.30

alert = int(share > threshold)

with open(
    "reports/day12_prometheus_alert.txt",
    "w"
) as f:

    f.write(
        f"data_drift_share {share:.4f}\n"
    )

    f.write(
        f"data_drift_alert {alert}\n"
    )

# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n" + "=" * 60)

print("DAY 12 — DATA DRIFT MONITORING")

print("=" * 60)

for r in drift_results:

    print(
        f"{r['feature']} → "
        f"p-value={r['p_value']:.6f} | "
        f"Drift={r['drift_detected']}"
    )

print("\n")

print(
    f"🎯 Drifted features: "
    f"{n_drifted}/{len(features)} "
    f"(share={share:.2f})"
)

if alert:

    print(
        "🚨 ALERT: Drift threshold exceeded!"
    )

else:

    print(
        "✅ Drift within acceptable range"
    )

print("\n📄 Alert metrics saved:")
print("reports/day12_prometheus_alert.txt")

Sales Shape: (329530, 11)

Reference Rows: 20123
Current Rows  : 18639

DAY 12 — DATA DRIFT MONITORING
Quantity → p-value=0.770693 | Drift=False
UnitPrice → p-value=0.230922 | Drift=False
TotalPrice → p-value=0.362386 | Drift=False


🎯 Drifted features: 0/3 (share=0.00)
✅ Drift within acceptable range

📄 Alert metrics saved:
reports/day12_prometheus_alert.txt
